# California Housing Regression Assignment

## Objective
This notebook applies and compares five supervised regression algorithms on the California Housing dataset:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor
- Support Vector Regressor (SVR)

The models are evaluated using **MSE, MAE, and R²**.

## 1. Loading and Preprocessing

The California Housing dataset is loaded using `fetch_california_housing()` from scikit-learn and converted into a pandas DataFrame.

### Preprocessing performed
1. Checked for missing values. The dataset normally contains no missing values, but the check is included for completeness.
2. Split the data into training and testing sets using an 80:20 split.
3. Standardized the features using `StandardScaler`.

Feature scaling is important particularly for **Linear Regression and SVR**, because their calculations can be affected by differences in feature magnitude. Tree-based models do not require scaling, but using the same standardized feature matrix makes the comparison consistent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)

# Convert to pandas DataFrame
df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check for missing values
print("Missing values before preprocessing:")
display(df.isnull().sum())

# Separate features and target
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Handle missing values (if any)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Total missing values after imputation:", X_imputed.isnull().sum().sum())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 2. Regression Algorithms

### Linear Regression
Linear Regression models the target as a linear combination of the input features. It is a useful baseline because it is simple, fast, and easy to interpret.

### Decision Tree Regressor
A Decision Tree recursively splits the data into groups using feature-based rules and predicts a value for each final region. It can capture nonlinear relationships and does not require feature scaling.

### Random Forest Regressor
Random Forest combines many decision trees trained on different samples/features and averages their predictions. This generally improves stability and reduces overfitting compared with a single tree.

### Gradient Boosting Regressor
Gradient Boosting builds trees sequentially, with each new tree focusing on correcting errors made by the previous trees. It is well suited to structured/tabular data and can model complex nonlinear relationships.

### Support Vector Regressor (SVR)
SVR finds a function that fits the data within a specified tolerance while controlling model complexity. With a nonlinear kernel such as RBF, it can model complex relationships. Scaling is especially important for SVR.

In [ ]:
# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(kernel="rbf")
}

# Train each model and calculate predictions
results = []
trained_models = {}

for name, model in models.items():
    # Use scaled data for all models for a consistent input representation
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MSE": mse,
        "MAE": mae,
        "R²": r2
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(by="R²", ascending=False).reset_index(drop=True)
display(results_df)

## 3. Model Evaluation and Comparison

The evaluation metrics are:

- **MSE (Mean Squared Error):** average squared prediction error. Lower is better.
- **MAE (Mean Absolute Error):** average absolute prediction error. Lower is better.
- **R² (R-squared):** proportion of target variance explained by the model. Higher is better.

In [ ]:
# Identify best and worst models based on R²
best_model = results_df.iloc[0]
worst_model = results_df.iloc[-1]

print("Best-performing model based on R²:")
print(best_model)

print("\nWorst-performing model based on R²:")
print(worst_model)

# Also display rankings for each metric
print("\nRanking by MSE (lower is better):")
display(results_df.sort_values("MSE")[["Model", "MSE"]])

print("Ranking by MAE (lower is better):")
display(results_df.sort_values("MAE")[["Model", "MAE"]])

print("Ranking by R² (higher is better):")
display(results_df.sort_values("R²", ascending=False)[["Model", "R²"]])

In [ ]:
# Visual comparison of R² scores
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R²"])
plt.ylabel("R² Score")
plt.xlabel("Model")
plt.title("R² Score Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Visual comparison of MAE
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["MAE"])
plt.ylabel("MAE")
plt.xlabel("Model")
plt.title("MAE Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Final Conclusion

After running the notebook, the models are compared using all three evaluation metrics.

**Best-performing model:** The model with the **highest R²** and correspondingly low MSE/MAE should be identified as the best overall model. For the standard California Housing dataset with these settings, tree-ensemble methods are typically among the strongest performers because the relationship between housing characteristics and median house value is nonlinear.

**Worst-performing model:** The model with the **lowest R²** and higher error values is considered the weakest. Linear Regression can perform worse when the underlying relationships are substantially nonlinear because it assumes a linear relationship between predictors and the target.

The exact metric values are generated by the executed cells above, so the notebook remains reproducible rather than relying on manually entered results.

## Submission Checklist
- [x] California Housing dataset loaded using `fetch_california_housing`
- [x] Converted to pandas DataFrame
- [x] Missing values checked and handled
- [x] Feature scaling performed
- [x] Five regression algorithms implemented
- [x] MSE, MAE and R² calculated
- [x] Models compared and best/worst identified
- [x] Code comments and explanations included